<a href="https://colab.research.google.com/github/2Bert0/previsao-atraso-entregas-ml/blob/main/Previs%C3%A3o_entrega_pedido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [2]:
import os
import pandas as pd

orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))

# Filtra pedidos efetivamente entregues com data estimada válida
entregues = orders.dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"]
).copy()

entregues["data_entrega"] = pd.to_datetime(
    entregues["order_delivered_customer_date"]
)
entregues["data_estimada"] = pd.to_datetime(
    entregues["order_estimated_delivery_date"]
)

# 1 = atrasou (entregue após a data estimada), 0 = no prazo ou antecipado
entregues["atrasou"] = (
    entregues["data_entrega"] > entregues["data_estimada"]
).astype(int)

print("Total de pedidos entregues:", len(entregues))
print("Proporção de atrasos (%):")
print((entregues["atrasou"].value_counts(normalize=True) * 100).round(2))

Total de pedidos entregues: 96476
Proporção de atrasos (%):
atrasou
0    91.89
1     8.11
Name: proportion, dtype: float64


In [3]:
sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv"))

# Junta pedidos com itens (1 linha por pedido para evitar duplicar pedidos no target)
# Pegamos o total de itens, soma de preço, soma de frete e o produto principal
itens_agg = (
    items.groupby("order_id", as_index=False)
    .agg(
        qtd_itens=("order_item_id", "count"),
        valor_produtos=("price", "sum"),
        valor_frete=("freight_value", "sum"),
        product_id=("product_id", "first"),
        seller_id=("seller_id", "first"),
    )
)

df = entregues.merge(itens_agg, on="order_id", how="inner")
df = df.merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
)
df = df.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left"
)
df = df.merge(
    products[
        [
            "product_id",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ]
    ],
    on="product_id",
    how="left",
)

# Features de tempo disponíveis no momento da compra
df["data_compra"] = pd.to_datetime(df["order_purchase_timestamp"])
df["prazo_estimado_dias"] = (
    df["data_estimada"] - df["data_compra"]
).dt.total_seconds() / 86400
df["dia_semana_compra"] = df["data_compra"].dt.dayofweek
df["mes_compra"] = df["data_compra"].dt.month

# Features de rota e proporção
df["mesma_uf"] = (df["customer_state"] == df["seller_state"]).astype(int)
df["proporcao_frete"] = df["valor_frete"] / (df["valor_produtos"] + 1e-5)

features = [
    "prazo_estimado_dias",
    "dia_semana_compra",
    "mes_compra",
    "qtd_itens",
    "valor_produtos",
    "valor_frete",
    "proporcao_frete",
    "mesma_uf",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

modelo_df = df[features + ["atrasou"]].copy()
print("Shape do dataset:", modelo_df.shape)
print("\nValores nulos por coluna:")
print(modelo_df.isnull().sum()[modelo_df.isnull().sum() > 0])

Shape do dataset: (96476, 13)

Valores nulos por coluna:
product_weight_g     16
product_length_cm    16
product_height_cm    16
product_width_cm     16
dtype: int64
